In [ ]:
!pip install gradio pandas scikit-learn matplotlib

In [ ]:
import gradio as gr
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from datetime import datetime, timedelta
import random

# ---------------- ML MODEL ----------------
data = {
    "Score_Percentage": [30, 40, 50, 60, 70, 80, 90],
    "Hours": [4, 3.5, 3, 2.5, 2, 1.5, 1]
}

df = pd.DataFrame(data)
model = LinearRegression()
model.fit(df[["Score_Percentage"]], df["Hours"])


# ---------------- CORE FUNCTION ----------------
def generate_dashboard(name, exam_type, total_marks, start_time,
                       s1, m1, s2, m2, s3, m3, s4, m4, s5, m5):

    subjects = [s1, s2, s3, s4, s5]
    marks = [m1, m2, m3, m4, m5]

    clean_subjects = []
    clean_marks = []

    for sub, mark in zip(subjects, marks):
        if sub and sub.strip() != "":
            clean_subjects.append(sub.strip())
            clean_marks.append(mark)

    if len(clean_subjects) == 0:
        return (
            gr.update(visible=False),
            "Please enter at least one subject.",
            None,
            None,
            None
        )

    percentages = [(m / total_marks) * 100 for m in clean_marks]

    input_df = pd.DataFrame({"Score_Percentage": percentages})
    predicted_hours = model.predict(input_df)
    hours = [max(1, round(h, 2)) for h in predicted_hours]

    avg_percentage = round(sum(percentages) / len(percentages), 2)
    total_hours = round(sum(hours), 2)

    summary = f"""
Student Name: {name}
Exam Type: {exam_type}
Average Percentage: {avg_percentage}%
Total Recommended Study Hours: {total_hours}
Subjects Count: {len(clean_subjects)}
"""

    # -------- BAR GRAPH --------
    fig1, ax1 = plt.subplots()
    ax1.bar(clean_subjects, hours)
    ax1.set_title("Study Hours per Subject")
    ax1.set_ylabel("Hours")

    # -------- PIE GRAPH --------
    fig2, ax2 = plt.subplots()
    ax2.pie(hours, labels=clean_subjects, autopct='%1.1f%%')
    ax2.set_title("Study Time Distribution")

    # -------- TIMETABLE --------
    try:
        current_time = datetime.strptime(start_time, "%H:%M")
    except:
        current_time = datetime.strptime("09:00", "%H:%M")

    timetable = []

    for i in range(len(clean_subjects)):
        study_minutes = int(hours[i] * 60)

        start = current_time.strftime("%I:%M %p")
        end_time = current_time + timedelta(minutes=study_minutes)
        end = end_time.strftime("%I:%M %p")

        timetable.append([start, end, f"Study - {clean_subjects[i]}"])
        current_time = end_time

        if i != len(clean_subjects) - 1:
            break_start = current_time.strftime("%I:%M %p")
            break_end_time = current_time + timedelta(minutes=10)
            break_end = break_end_time.strftime("%I:%M %p")

            timetable.append([break_start, break_end, "Break (10 minutes)"])
            current_time = break_end_time

    timetable_df = pd.DataFrame(
        timetable,
        columns=["Start Time", "End Time", "Activity"]
    )

    return (
        gr.update(visible=True),
        summary,
        fig1,
        fig2,
        timetable_df
    )


# ---------------- PROFESSIONAL CSS ----------------
css = """
.gradio-container {
    max-width: 1100px !important;
    margin: auto;
}
"""


# ---------------- UI ----------------
with gr.Blocks(
    theme=gr.themes.Soft(),   # ✅ Professional theme
    css=css,
    title="AI Study Planner"
) as app:

    gr.Markdown("# AI Study Planner")
    gr.Markdown("### Smart Study Recommendations Based on Performance")

    with gr.Row():

        # -------- INPUT PANEL --------
        with gr.Column(scale=1):
            gr.Markdown("## Student Details")

            name = gr.Textbox(label="Student Name")

            exam_type = gr.Dropdown(
                choices=["Mids", "Semester", "Final Exam", "Custom"],
                label="Type of Exam",
                value="Semester"
            )

            total_marks = gr.Number(
                label="Total Marks per Subject",
                value=100
            )

            start_time = gr.Textbox(
                label="Study Start Time (HH:MM)",
                value="09:00"
            )

            gr.Markdown("## Subjects & Marks")

            with gr.Row():
                s1 = gr.Textbox(label="Subject 1")
                m1 = gr.Number(label="Marks", value=50)

            with gr.Row():
                s2 = gr.Textbox(label="Subject 2")
                m2 = gr.Number(label="Marks", value=50)

            with gr.Row():
                s3 = gr.Textbox(label="Subject 3")
                m3 = gr.Number(label="Marks", value=50)

            with gr.Row():
                s4 = gr.Textbox(label="Subject 4")
                m4 = gr.Number(label="Marks", value=50)

            with gr.Row():
                s5 = gr.Textbox(label="Subject 5")
                m5 = gr.Number(label="Marks", value=50)

            generate_btn = gr.Button("Generate Study Plan")

        # -------- OUTPUT PANEL --------
        with gr.Column(scale=1):
            gr.Markdown("## Study Dashboard")

            summary_box = gr.Textbox(
    label="Academic Overview",
    lines=10,
    interactive=False,
    show_copy_button=True
)

            bar_plot = gr.Plot()
            pie_plot = gr.Plot()

            timetable = gr.Dataframe(label="Daily Study Timetable")

    generate_btn.click(
        fn=generate_dashboard,
        inputs=[name, exam_type, total_marks, start_time,
                s1, m1, s2, m2, s3, m3, s4, m4, s5, m5],
        outputs=[summary_box, summary_box, bar_plot, pie_plot, timetable]
    )


# ✅ Auto Port
port = random.randint(7860, 9000)
app.launch(server_name="0.0.0.0", server_port=port)

/tmp/ipython-input-274/245249174.py:121: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(
/tmp/ipython-input-274/245249174.py:121: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://030cd4fdf8e8b1bb40.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
